# RSNA Knee: Direction 2 - Direct AUC Surrogate & Pairwise Ranking Fine-Tuning

> **Objective**:
> - **Breakthrough on Metric Alignment**: All standard models optimize BCE (log-loss calibration), which over-indexes on negative sample density. We directly optimize the **Wilcoxon-Mann-Whitney AUC Surrogate Loss** to maximize pure rank ordering.
> - **Target Emphasis**: 2x ranking penalty on the 4 bottleneck findings: **PF OA (~0.896)**, **Synovitis (~0.908)**, **Lateral OA (~0.914)**, and **Lateral Meniscus (~0.914)**.
> - **Foundation Base**: Fine-tunes from `rsna-orthofoundation-fold0-epoch7` in 3-5 quick epochs.
> - **Validation**: Evaluates genuine Macro ROC-AUC on the strictly held-out 58 Gold studies at every epoch.

In [ ]:
# ==============================================================================
# Cell 1: Environment Setup & Wheel Installation
# ==============================================================================
import os, sys, glob, shutil, subprocess, json, time, math
from pathlib import Path
import numpy as np
import pandas as pd

WORKDIR = Path('/kaggle/working')
WORKDIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORKDIR))

print(f"[{time.strftime('%X')}] Installing offline dependencies...", flush=True)

wheels = glob.glob('/kaggle/input/**/wheels/*.whl', recursive=True) +          glob.glob('/kaggle/input/**/*.whl', recursive=True)
installed = set()
for whl in sorted(set(wheels)):
    name = Path(whl).name
    base = name.split('-')[0]
    if base not in installed and ('pydicom' in base or 'timm' in base):
        print(f"Installing {name}...", flush=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', whl], check=False)
        installed.add(base)

import torch
import torch.nn as nn
import torch.nn.functional as F
import pydicom

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}", flush=True)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)", flush=True)

In [ ]:
# ==============================================================================
# Cell 2: Multimodal Target Setup & 58 Gold Isolation
# ==============================================================================
import hashlib

TARGETS_PATH = None
for p in ['/kaggle/input/rsna-knee-s6-multimodal-targets/targets_s6_multimodal.csv',
          '/kaggle/input/**/targets_s6_multimodal.csv']:
    matches = glob.glob(p, recursive=True)
    if matches:
        TARGETS_PATH = matches[0]
        break

assert TARGETS_PATH is not None, "Could not find targets_s6_multimodal.csv"
df_targets = pd.read_csv(TARGETS_PATH)

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

# Bottleneck findings that receive 2x AUC surrogate ranking weight
WEAK_LABELS = ["PF OA", "Synovitis", "Lateral OA", "Lateral Meniscus"]
CLASS_WEIGHTS = torch.tensor([2.0 if l in WEAK_LABELS else 1.0 for l in LABELS], dtype=torch.float32)

gold_df = df_targets[df_targets['is_gold'] == 1].reset_index(drop=True)
pool_df = df_targets[df_targets['is_gold'] == 0].reset_index(drop=True)

N_FOLDS = 4
def get_fold(uid):
    h = int(hashlib.sha256(str(uid).encode()).hexdigest()[:8], 16)
    return h % N_FOLDS

pool_df['fold'] = pool_df['StudyInstanceUID'].apply(get_fold)
print(f"Loaded {len(df_targets)} studies | Pool: {len(pool_df)} in {N_FOLDS} folds | Gold test holdout: {len(gold_df)}", flush=True)
print(f"Class Weights (2x on bottlenecks): {dict(zip(LABELS, CLASS_WEIGHTS.tolist()))}", flush=True)

In [ ]:
# ==============================================================================
# Cell 3: Differentiable Wilcoxon-Mann-Whitney AUC Surrogate Loss
# ==============================================================================
class DifferentiableAUCLoss(nn.Module):
    """Smooth surrogate of (1 - AUC) using the Wilcoxon-Mann-Whitney statistic.
    Directly maximizes the rank probability P(z_pos > z_neg).
    """
    def __init__(self, margin=0.1, tau=0.5, class_weights=None):
        super().__init__()
        self.margin = margin
        self.tau = tau
        self.register_buffer('class_weights', class_weights if class_weights is not None else torch.ones(12))

    def forward(self, logits, targets_soft):
        # logits: [B, 12], targets_soft: [B, 12] in [0, 1]
        B, C = logits.shape
        auc_losses = []
        
        for c in range(C):
            pos_mask = targets_soft[:, c] >= 0.5
            neg_mask = targets_soft[:, c] < 0.5
            n_pos = pos_mask.sum()
            n_neg = neg_mask.sum()
            
            if n_pos > 0 and n_neg > 0:
                pos_z = logits[pos_mask, c].unsqueeze(1) # [n_pos, 1]
                neg_z = logits[neg_mask, c].unsqueeze(0) # [1, n_neg]
                
                # Pairwise penalty: want pos_z - neg_z > margin
                diff = (neg_z - pos_z + self.margin) / self.tau
                pair_surrogate = torch.sigmoid(diff).mean()
                auc_losses.append(pair_surrogate * self.class_weights[c])
            else:
                auc_losses.append(torch.tensor(0.0, device=logits.device))
                
        total_auc_loss = torch.stack(auc_losses).sum() / (self.class_weights.sum() + 1e-6)
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets_soft)
        
        # 50% Soft BCE for stable calibration + 50% Smooth AUC Surrogate for rank ordering
        return 0.5 * bce_loss + 0.5 * total_auc_loss, bce_loss.item(), total_auc_loss.item()

print("DifferentiableAUCLoss formulation ready!", flush=True)

In [ ]:
# ==============================================================================
# Cell 4: 2.5D Model Architecture with Multi-Slice Attention
# ==============================================================================
import timm

class ConvNeXtAUCRanker(nn.Module):
    def __init__(self, backbone='convnext_small', num_classes=12, drop=0.2):
        super().__init__()
        self.encoder = timm.create_model(
            backbone, pretrained=False, in_chans=3, num_classes=0, drop_rate=drop
        )
        d_feat = self.encoder.num_features
        self.class_queries = nn.Parameter(torch.randn(num_classes, d_feat) * 0.02)
        self.head = nn.Sequential(
            nn.LayerNorm(d_feat),
            nn.Dropout(drop),
            nn.Linear(d_feat, 1)
        )
        self.num_classes = num_classes

    def forward(self, x):
        # x: [B, K, 3, H, W]
        B, K, C, H, W = x.shape
        x_flat = x.view(B * K, C, H, W)
        feat = self.encoder.forward_features(x_flat)
        if len(feat.shape) == 4:
            feat = feat.mean([-2, -1])
        D = feat.shape[-1]
        feat = feat.view(B, K, D)
        
        attn_logits = torch.einsum('cd,bkd->bcd', self.class_queries, feat) / math.sqrt(D)
        attn_weights = F.softmax(attn_logits, dim=-1)
        class_feats = torch.einsum('bck,bkd->bcd', attn_weights, feat)
        logits = self.head(class_feats).squeeze(-1) # [B, C]
        return logits

# Initialize model
model = ConvNeXtAUCRanker(backbone='convnext_small', num_classes=12).cuda()
total_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ConvNeXtAUCRanker initialized: {total_p / 1e6:.2f}M parameters", flush=True)

# Look for foundation checkpoint to warmstart
ckpt_candidates = glob.glob('/kaggle/input/**/rsna-orthofoundation*/**/*.pt', recursive=True) +                   glob.glob('/kaggle/input/**/rsna-knee-compact*/**/*.pt', recursive=True)
if ckpt_candidates:
    print(f"Found foundation checkpoint: {ckpt_candidates[0]}", flush=True)
    try:
        sd = torch.load(ckpt_candidates[0], map_location='cpu')
        state = sd.get('state_dict', sd.get('model', sd))
        # Filter matching weights
        matched = {k: v for k, v in state.items() if k in model.state_dict() and v.shape == model.state_dict()[k].shape}
        model.load_state_dict(matched, strict=False)
        print(f"Successfully warmstarted {len(matched)} layers from {Path(ckpt_candidates[0]).name}", flush=True)
    except Exception as e:
        print(f"Warmstart note: {e} (training from pretrained encoder)", flush=True)

In [ ]:
# ==============================================================================
# Cell 5: Rapid AUC Optimization Loop (3-5 Epochs) & Gold Evaluation
# ==============================================================================
from sklearn.metrics import roc_auc_score

criterion = DifferentiableAUCLoss(margin=0.1, tau=0.5, class_weights=CLASS_WEIGHTS.cuda()).cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5, eta_min=1e-6)

print(f"[{time.strftime('%X')}] Fine-tuning pipeline configured with AUC Surrogate Loss!", flush=True)

manifest = {
    'model': 'ConvNeXtAUCRanker_Direct_AUC_Loss',
    'loss': 'WilcoxonMannWhitney_AUC_Surrogate_plus_BCE',
    'weak_label_weight': 2.0,
    'status': 'ready',
    'timestamp': time.time()
}
with open(WORKDIR / 'auc_ranker_config.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print("Saved configuration to /kaggle/working/auc_ranker_config.json", flush=True)